importing packages

In [2]:
import pandas as pd 
import numpy as np 
import os
from matplotlib import pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score

from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

import xgboost as xgboost
import pickle

In [3]:
#base directori

base_dir = "Telecommunications_Industry/csv files/"
csv_dir = "created csv"
model_dir = "saved models"


Reading data 
listing all col in tables

In [4]:
list_of_files = os.listdir(base_dir)


set_of_columns_awailable = set()

for file in list_of_files:

    if ".csv" in file:

        df = pd.read_csv(base_dir + file) 
        cols_in_df = df.columns.tolist() 

        set_of_columns_awailable.update(cols_in_df)
        print("columns in file :", file ,"are" , cols_in_df)
        print()

        print("Total unique columns availabel in all files : " , len(set_of_columns_awailable))
        print(set_of_columns_awailable)
        


columns in file : CustomerChurn.csv are ['LoyaltyID', 'Customer ID', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn']

Total unique columns availabel in all files :  21
{'Multiple Lines', 'Tech Support', 'Paperless Billing', 'Customer ID', 'Senior Citizen', 'Contract', 'Online Backup', 'LoyaltyID', 'Online Security', 'Churn', 'Partner', 'Monthly Charges', 'Device Protection', 'Streaming Movies', 'Phone Service', 'Streaming TV', 'Internet Service', 'Tenure', 'Dependents', 'Payment Method', 'Total Charges'}
columns in file : Telco_customer_churn_services.csv are ['Service ID', 'Customer ID', 'Count', 'Quarter', 'Referred a Friend', 'Number of Referrals', 'Tenure in Months', 'Offer', 'Phone Service', 'Avg Monthly Long Distance Ch

combining all col to creating overall the dataset to treating for preparing several models in import

In [13]:
df = pd.read_csv(base_dir + "Telco_customer_churn.csv")   # reading all data into single dataframe
df = df.rename(columns={'CostemerID':'Customer ID'})  # renaming column to have same name across all files

list_of_csvs = ['CustomerChurn.csv', 
                'Telco_customer_churn_demographics.csv',
                'Telco_customer_churn_location.csv',
                'Telco_customer_churn_population.csv',
                'Telco_customer_churn_services.csv',
                'Telco_customer_churn_status.csv']

for file in list_of_csvs: 
    temp = pd.read_csv(base_dir + file) 
    # Check if both dataframes have 'Customer ID' before merging
    if 'Customer ID' in temp.columns.tolist() and 'Customer ID' in df.columns.tolist():
        df = pd.merge(df, temp, on='Customer ID', how='left', suffixes=('', '_remove'))
    elif 'Zip code' in temp.columns.tolist() and 'Zip code' in df.columns.tolist():
        df = pd.merge(df, temp, on="Zip code", how='left', suffixes=('', '_remove'))
    else:
        print(f"Skipping file {file}: no common key column found for merging.")

# Drop columns with '_remove' in their name
df.drop([i for i in df.columns if 'remove' in i], axis=1, inplace=True)

print("Total Number of Columns : ", len(df.columns))
print("List of columns :", df.columns.tolist())
df.head()


Skipping file CustomerChurn.csv: no common key column found for merging.
Skipping file Telco_customer_churn_demographics.csv: no common key column found for merging.
Skipping file Telco_customer_churn_location.csv: no common key column found for merging.
Skipping file Telco_customer_churn_population.csv: no common key column found for merging.
Skipping file Telco_customer_churn_services.csv: no common key column found for merging.
Skipping file Telco_customer_churn_status.csv: no common key column found for merging.
Total Number of Columns :  33
List of columns : ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn L

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [15]:
df.info()
df.to_csv(csv_dir + "Telecom_customer_churn_complete.csv")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

DATA Preprocessing 
Dealing with null values 

In [16]:
columns_with_null_values = [(col, df[col].isnull().sum()) for col in df.columns.tolist() if df[col].isnull().sum() > 0]
columns_with_null_values

[('Churn Reason', np.int64(5174))]

In [18]:
# replacing na values in "Churn Category" with "Not Applicable" if the column exists
if "Churn Category" in df.columns:
    df["Churn Category"].fillna("Not Applicable", inplace=True)
else:
    print("Column 'Churn Category' not found in dataframe.")

# replacing na values in "Churn Reason" with "Not Churned" if the column exists
if "Churn Reason" in df.columns:
    df["Churn Reason"].fillna("Not Churned", inplace=True)
else:
    print("Column 'Churn Reason' not found in dataframe.")

Column 'Churn Category' not found in dataframe.


/tmp/ipykernel_5269/2264752143.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Churn Reason"].fillna("Not Churned", inplace=True)


In [19]:
#data type of total charges column is object - as it contains null values as blank space strings
#we need to replace that with 0.0
df["Total Charges"] = np.where(df["Total Charges"] == " ", '0.0', df["Total Charges"])
df["Total Charges"] = df["Total Charges"].astype(float)

Datawrangling


In [ ]:
for col in df.columns.tolist():
    print(col, ":", df[col].dtype)
    print("number of unique values :", df[col].nunique())
    print("Unique Values :", df[col].unique())
    print("Unlike values :", df[col].unique()[:10])

    if df[col].dtype == 'int64' or df[col].dtype == 'float64':
        print("Statistical Summary :")
        print(df[col].describe())
        print("max values in indexes :", df[col].sort_values(ascending=False).head(10).index.tolist())
        print("min values in indexes by order :", df[col].sort_values(ascending=True).head(10).index.tolist())
        print()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4214140387.py, line 2)